In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier

# ===== Config =====
SEED = 42
rng = np.random.default_rng(SEED)
N_SPLITS = 5
N_TRIALS_MAX = 5000
MAX_HOURS = 0.0  # set 0.0 for one-trial dry run
USE_EARLY_STOPPING = True
EARLY_STOPPING_ROUNDS = 100
VERBOSE_EVERY = 100
SAVE_EVERY = 100

# ===== Paths =====
local_train = '/workspaces/KagglePredictingStellarClass/data/train.csv'
kaggle_train = '/kaggle/input/competitions/playground-series-s6e6/train.csv'
train_path = local_train if os.path.exists(local_train) else kaggle_train
if not os.path.exists(train_path):
    raise FileNotFoundError('Could not find train.csv in local or Kaggle path.')

print('Train file:', train_path)

# ===== Top 4 coarse anchors (hardcoded) =====
top4 = pd.DataFrame([
    {
        'trial': 223,
        'mean_balanced_accuracy': 0.9465704168303654,
        'n_estimators': 3816,
        'learning_rate': 0.07263498525089532,
        'max_depth': 8,
        'min_child_weight': 4.806702051952081,
        'subsample': 0.787060980980085,
        'colsample_bytree': 0.6678856420394242,
        'reg_alpha': 0.9612993507910456,
        'reg_lambda': 0.020114947651884443,
        'gamma': 0.7676577316852373
    },
    {
        'trial': 156,
        'mean_balanced_accuracy': 0.9465097228338951,
        'n_estimators': 2954,
        'learning_rate': 0.054855608503623095,
        'max_depth': 9,
        'min_child_weight': 4.968345551700789,
        'subsample': 0.622108163800491,
        'colsample_bytree': 0.7429478494008217,
        'reg_alpha': 0.0021088023972270103,
        'reg_lambda': 1.1283019447834943,
        'gamma': 1.0269514883620334
    },
    {
        'trial': 1112,
        'mean_balanced_accuracy': 0.9465033856562829,
        'n_estimators': 3485,
        'learning_rate': 0.015048477769496603,
        'max_depth': 9,
        'min_child_weight': 9.619481980620714,
        'subsample': 0.8825551519246035,
        'colsample_bytree': 0.9007947552673445,
        'reg_alpha': 0.0016735622211771458,
        'reg_lambda': 0.0928917784004792,
        'gamma': 0.5987541360360948
    },
    {
        'trial': 478,
        'mean_balanced_accuracy': 0.9465013072314816,
        'n_estimators': 3973,
        'learning_rate': 0.049175628575270205,
        'max_depth': 6,
        'min_child_weight': 8.54413492386206,
        'subsample': 0.9363777869195118,
        'colsample_bytree': 0.907406797039591,
        'reg_alpha': 0.1376450123063579,
        'reg_lambda': 0.49172693348586555,
        'gamma': 0.15542090004015185
    }
])

print('Using hardcoded top 4 coarse trials as anchors for fine search.')
print(top4[['trial','mean_balanced_accuracy','n_estimators','learning_rate','max_depth','min_child_weight','subsample','colsample_bytree','reg_alpha','reg_lambda','gamma']])

# ===== Build fine-search ranges around each anchor row =====
def _clip(v, lo, hi):
    return float(min(max(v, lo), hi))

def build_ranges(anchor_row):
    return {
        'n_estimators': (max(300, int(anchor_row['n_estimators'] * 0.8)), int(anchor_row['n_estimators'] * 1.2)),
        'learning_rate': (_clip(anchor_row['learning_rate'] * 0.75, 0.005, 0.3), _clip(anchor_row['learning_rate'] * 1.25, 0.005, 0.3)),
        'max_depth': (max(3, int(anchor_row['max_depth']) - 1), min(12, int(anchor_row['max_depth']) + 1)),
        'min_child_weight': (_clip(anchor_row['min_child_weight'] * 0.75, 0.5, 20.0), _clip(anchor_row['min_child_weight'] * 1.25, 0.5, 20.0)),
        'subsample': (_clip(anchor_row['subsample'] - 0.10, 0.5, 1.0), _clip(anchor_row['subsample'] + 0.10, 0.5, 1.0)),
        'colsample_bytree': (_clip(anchor_row['colsample_bytree'] - 0.10, 0.5, 1.0), _clip(anchor_row['colsample_bytree'] + 0.10, 0.5, 1.0)),
        'reg_alpha': (max(1e-6, float(anchor_row['reg_alpha']) / 3.0), max(1e-5, float(anchor_row['reg_alpha']) * 3.0)),
        'reg_lambda': (max(1e-6, float(anchor_row['reg_lambda']) / 3.0), max(1e-5, float(anchor_row['reg_lambda']) * 3.0)),
        'gamma': (_clip(float(anchor_row['gamma']) - 1.0, 0.0, 10.0), _clip(float(anchor_row['gamma']) + 1.0, 0.0, 10.0))
    }

# ===== Data preprocessing (same as coarse) =====
df = pd.read_csv(train_path)
df_eng = df.drop(columns=['id']).copy()
df_eng['u_g'] = df_eng['u'] - df_eng['g']
df_eng['g_r'] = df_eng['g'] - df_eng['r']
df_eng['r_i'] = df_eng['r'] - df_eng['i']
df_eng['i_z'] = df_eng['i'] - df_eng['z']
df_eng = df_eng.drop(columns=['u', 'g', 'r', 'i', 'z'])

for col in ['spectral_type', 'galaxy_population']:
    df_eng[col] = LabelEncoder().fit_transform(df_eng[col].astype(str))

X = df_eng.drop(columns=['class'])
y = LabelEncoder().fit_transform(df_eng['class'])
num_class = len(np.unique(y))
print('X shape:', X.shape)
print('Number of classes:', num_class)

# ===== Output setup =====
out_dir = '/kaggle/working/fine_search_outputs' if os.path.exists('/kaggle/working') else '/workspaces/KagglePredictingStellarClass/fine_search_outputs'
os.makedirs(out_dir, exist_ok=True)
csv_path = os.path.join(out_dir, 'xgb_fine_random_search_results.csv')
json_path = os.path.join(out_dir, 'xgb_fine_random_search_best.json')
ranges_path = os.path.join(out_dir, 'xgb_fine_search_ranges.json')
with open(ranges_path, 'w', encoding='utf-8') as f:
    json.dump(top4.to_dict(orient='records'), f, indent=2)

def sample_params_fine(rng_local):
    anchor_idx = int(rng_local.integers(0, len(top4)))
    anchor = top4.iloc[anchor_idx]
    ranges = build_ranges(anchor)
    ra_lo, ra_hi = ranges['reg_alpha']
    rl_lo, rl_hi = ranges['reg_lambda']
    return {
        'anchor_trial': int(anchor['trial']),
        'n_estimators': int(rng_local.integers(ranges['n_estimators'][0], ranges['n_estimators'][1] + 1)),
        'learning_rate': float(rng_local.uniform(ranges['learning_rate'][0], ranges['learning_rate'][1])),
        'max_depth': int(rng_local.integers(ranges['max_depth'][0], ranges['max_depth'][1] + 1)),
        'min_child_weight': float(rng_local.uniform(ranges['min_child_weight'][0], ranges['min_child_weight'][1])),
        'subsample': float(rng_local.uniform(ranges['subsample'][0], ranges['subsample'][1])),
        'colsample_bytree': float(rng_local.uniform(ranges['colsample_bytree'][0], ranges['colsample_bytree'][1])),
        'reg_alpha': float(np.exp(rng_local.uniform(np.log(ra_lo), np.log(ra_hi)))),
        'reg_lambda': float(np.exp(rng_local.uniform(np.log(rl_lo), np.log(rl_hi)))),
        'gamma': float(rng_local.uniform(ranges['gamma'][0], ranges['gamma'][1])),
        'n_jobs': -1
    }

def save_results(results_list):
    if not results_list:
        return
    tmp_df = pd.DataFrame(results_list).sort_values('mean_balanced_accuracy', ascending=False).reset_index(drop=True)
    tmp_df.to_csv(csv_path, index=False)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(tmp_df.iloc[0].to_dict(), f, indent=2)

# ===== Fine random search =====
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
results = []
overall_start = time.perf_counter()
max_seconds = MAX_HOURS * 3600.0

if MAX_HOURS <= 0:
    print('Dry run mode: MAX_HOURS <= 0, will run exactly 1 trial.')
else:
    print(f'Time budget: {MAX_HOURS:.2f} hours')

trial = 1
while trial <= N_TRIALS_MAX:
    elapsed_before_trial = time.perf_counter() - overall_start
    if MAX_HOURS > 0 and elapsed_before_trial >= max_seconds:
        print('Time budget reached before starting next trial. Stopping.')
        break

    params = sample_params_fine(rng)
    anchor_trial = int(params['anchor_trial'])
    params = {k: v for k, v in params.items() if k != 'anchor_trial'}
    fold_scores = []
    fit_times = []
    pred_times = []
    trial_start = time.perf_counter()

    for tr_idx, va_idx in skf.split(X, y):
        X_tr = X.iloc[tr_idx]
        y_tr = y[tr_idx]
        X_va = X.iloc[va_idx]
        y_va = y[va_idx]

        common = dict(
            objective='multi:softprob',
            num_class=num_class,
            eval_metric='mlogloss',
            tree_method='hist',
            random_state=SEED
        )
        if USE_EARLY_STOPPING:
            common['early_stopping_rounds'] = EARLY_STOPPING_ROUNDS

        fit_start = time.perf_counter()
        try:
            model = XGBClassifier(device='cuda', **params, **common)
            if USE_EARLY_STOPPING:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            else:
                model.fit(X_tr, y_tr, verbose=False)
        except Exception:
            model = XGBClassifier(device='cpu', **params, **common)
            if USE_EARLY_STOPPING:
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            else:
                model.fit(X_tr, y_tr, verbose=False)
        fit_elapsed = time.perf_counter() - fit_start

        pred_start = time.perf_counter()
        pred = model.predict(X_va)
        pred_elapsed = time.perf_counter() - pred_start

        score = balanced_accuracy_score(y_va, pred)
        fold_scores.append(float(score))
        fit_times.append(float(fit_elapsed))
        pred_times.append(float(pred_elapsed))

    trial_elapsed = time.perf_counter() - trial_start
    row = {
        'trial': trial,
        'anchor_trial': anchor_trial,
        'search_seed': SEED,
        'use_early_stopping': USE_EARLY_STOPPING,
        'early_stopping_rounds': EARLY_STOPPING_ROUNDS if USE_EARLY_STOPPING else 0,
        'mean_balanced_accuracy': float(np.mean(fold_scores)),
        'std_balanced_accuracy': float(np.std(fold_scores)),
        'fold_scores': fold_scores,
        'fit_time_sec_mean': float(np.mean(fit_times)),
        'pred_time_sec_mean': float(np.mean(pred_times)),
        'trial_time_sec': float(trial_elapsed),
        **params
    }
    results.append(row)

    elapsed_after_trial = time.perf_counter() - overall_start
    if trial == 1 or trial % VERBOSE_EVERY == 0:
        if MAX_HOURS > 0:
            remaining = max(0.0, max_seconds - elapsed_after_trial)
            print(
                f'Trial {trial} | mean_bal_acc={row["mean_balanced_accuracy"]:.6f} | '
                f'trial_time={trial_elapsed:.2f}s | elapsed={(elapsed_after_trial/3600):.2f}h | '
                f'remaining={(remaining/3600):.2f}h'
            )
        else:
            print(f'Trial {trial} | mean_bal_acc={row["mean_balanced_accuracy"]:.6f} | trial_time={trial_elapsed:.2f}s')

    if trial % SAVE_EVERY == 0 or trial == 1:
        save_results(results)

    if MAX_HOURS <= 0:
        print('Dry run complete (1 trial).')
        break

    if elapsed_after_trial >= max_seconds:
        print('Time budget reached after trial completion. Stopping.')
        break

    trial += 1

save_results(results)
if results:
    results_df = pd.DataFrame(results).sort_values('mean_balanced_accuracy', ascending=False).reset_index(drop=True)
    elapsed_total = time.perf_counter() - overall_start
    print('\n=== Fine Search Complete ===')
    print(f'Trials completed: {len(results)}')
    print(f'Total search time (minutes): {elapsed_total / 60:.2f}')
    print('Top 10 trials by balanced accuracy:')
    display(results_df.head(10))
    print('Saved fine results CSV:', csv_path)
    print('Saved fine best JSON:', json_path)
    print('Saved fine ranges JSON:', ranges_path)
else:
    print('No trials were completed. Nothing saved.')